# 🪙 CryptoAnalysisAgent - 加密货币多维分析智能体

## 项目介绍

本项目是一个基于 HelloAgents 框架构建的多 Agent 协作加密货币分析系统。

**核心理念**: 融合技术分析、链上数据、市场情绪三维视角，通过多 Agent 协作为交易决策提供结构化参考。

**设计灵感**: 来源于 [NOFXi](https://nofx.pro) 交易智能助手的 Skill-first 架构思想。

## 作者信息
- 项目: CryptoAnalysisAgent (灵感来源 NOFXi Trading)
- GitHub: @gpteth
- 日期: 2026-05-29

---
## Part 1: 环境配置

In [ ]:
# 安装依赖 (首次运行时取消注释)
# !pip install -q hello-agents[all] pandas numpy requests python-dotenv matplotlib

In [ ]:
import os
import sys
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

# 配置 LLM (如果没有 .env 文件，可以直接在这里设置)
# 推荐使用 ModelScope 的 Qwen2.5-72B 或 DeepSeek
os.environ.setdefault("LLM_MODEL_ID", "Qwen/Qwen2.5-72B-Instruct")
os.environ.setdefault("LLM_BASE_URL", "https://api-inference.modelscope.cn/v1/")
os.environ.setdefault("LLM_TIMEOUT", "60")

# ⚠️ 请在这里填入你的 API Key (或使用 .env 文件)
# os.environ["LLM_API_KEY"] = "your_api_key_here"

print("✅ 环境配置完成")
print(f"   模型: {os.environ.get('LLM_MODEL_ID')}")
print(f"   API: {os.environ.get('LLM_BASE_URL')}")

In [ ]:
# 将 src 目录加入路径
sys.path.insert(0, os.path.abspath('.'))

# 导入核心模块
from hello_agents import SimpleAgent, HelloAgentsLLM, ToolRegistry
from src.tools.technical import KlineFetchTool, TechnicalIndicatorTool, SupportResistanceTool
from src.tools.onchain import ExchangeFlowTool, WhaleActivityTool, ActiveAddressTool
from src.tools.sentiment import FearGreedTool, FundingRateTool, SocialSentimentTool
from src.agents.technical_agent import create_technical_agent
from src.agents.onchain_agent import create_onchain_agent
from src.agents.sentiment_agent import create_sentiment_agent
from src.agents.coordinator import create_coordinator

print("✅ 模块导入成功")

---
## Part 2: 工具演示

在构建 Agent 之前，先单独测试各个工具的功能。

### 2.1 技术分析工具

In [ ]:
# 测试 K 线数据获取
kline_tool = KlineFetchTool()
result = kline_tool.run({"symbol": "BTCUSDT", "interval": "4h", "limit": 20})
print(result)

In [ ]:
# 测试技术指标计算
indicator_tool = TechnicalIndicatorTool()
result = indicator_tool.run({"symbol": "BTCUSDT", "interval": "4h", "indicators": ["ema", "rsi", "macd", "boll", "atr"]})
print(result)

In [ ]:
# 测试支撑阻力位识别
sr_tool = SupportResistanceTool()
result = sr_tool.run({"symbol": "BTCUSDT", "interval": "4h"})
print(result)

### 2.2 链上数据工具

In [ ]:
# 测试交易所资金流向
flow_tool = ExchangeFlowTool()
result = flow_tool.run({"symbol": "BTC", "period": "7d"})
print(result)

In [ ]:
# 测试巨鲸活动
whale_tool = WhaleActivityTool()
result = whale_tool.run({"symbol": "BTC"})
print(result)

In [ ]:
# 测试活跃地址
address_tool = ActiveAddressTool()
result = address_tool.run({"symbol": "ETH"})
print(result)

### 2.3 情绪分析工具

In [ ]:
# 测试恐惧贪婪指数 (真实 API 数据)
fg_tool = FearGreedTool()
result = fg_tool.run({"days": 7})
print(result)

In [ ]:
# 测试资金费率 (真实 Binance 数据)
funding_tool = FundingRateTool()
result = funding_tool.run({"symbol": "BTCUSDT"})
print(result)

In [ ]:
# 测试社交情绪
social_tool = SocialSentimentTool()
result = social_tool.run({"symbol": "BTC"})
print(result)

---
## Part 3: 单 Agent 分析演示

展示每个专业 Agent 独立工作的效果。

In [ ]:
# 初始化 LLM
llm = HelloAgentsLLM()
print("✅ LLM 初始化完成")

### 3.1 技术分析 Agent

In [ ]:
# 创建技术分析 Agent
tech_agent = create_technical_agent(llm)

# 执行技术分析
print("=" * 60)
print("🔧 技术分析 Agent 工作中...")
print("=" * 60)

tech_result = tech_agent.run("请对 BTCUSDT 进行完整的技术分析，使用 4h 周期")
print(tech_result)

### 3.2 链上分析 Agent

In [ ]:
# 创建链上分析 Agent
onchain_agent = create_onchain_agent(llm)

# 执行链上分析
print("=" * 60)
print("⛓️ 链上分析 Agent 工作中...")
print("=" * 60)

onchain_result = onchain_agent.run("分析 BTC 的链上数据，包括资金流向、巨鲸活动和网络活跃度")
print(onchain_result)

### 3.3 情绪分析 Agent

In [ ]:
# 创建情绪分析 Agent
sent_agent = create_sentiment_agent(llm)

# 执行情绪分析
print("=" * 60)
print("💭 情绪分析 Agent 工作中...")
print("=" * 60)

sent_result = sent_agent.run("分析当前 BTC 市场的情绪状况，包括恐惧贪婪指数、资金费率和社交热度")
print(sent_result)

---
## Part 4: 多 Agent 协作 - 综合分析

这是本项目的核心功能：通过 Coordinator Agent 协调三个专业 Agent，
生成多维度交叉验证的综合分析报告。

In [ ]:
# 创建综合分析协调 Agent
# Coordinator 内部会自动调用技术、链上、情绪三个子 Agent
coordinator = create_coordinator(llm=llm)

print("✅ 综合分析系统就绪")
print("   - 技术分析师 ✓")
print("   - 链上分析师 ✓")
print("   - 情绪分析师 ✓")
print("   - 综合协调员 ✓")

In [ ]:
# 🎯 核心功能演示: BTC 综合分析
print("=" * 60)
print("🎯 启动 BTC 综合分析 (多 Agent 协作)")
print("=" * 60)
print()

comprehensive_report = coordinator.run(
    "请对 BTC 进行全面的综合分析。"
    "分别从技术面、链上面、情绪面三个维度进行分析，"
    "然后综合三个维度的结论，给出交叉验证后的综合判断和条件化建议。"
)

print(comprehensive_report)

In [ ]:
# 保存分析报告
os.makedirs("outputs", exist_ok=True)
with open("outputs/btc_analysis_report.md", "w", encoding="utf-8") as f:
    f.write(comprehensive_report)

print("\n📄 报告已保存至 outputs/btc_analysis_report.md")

---
## Part 4.5: Agent 考核 - 报告质量与性能指标

按四类指标对系统进行自动化考核:

| 指标 | 类型 | 实现 |
|---|---|---|
| 结构合规率 | 规则 | 检查报告是否包含模板要求的 8 个章节 |
| 条件化建议 | 规则 | 检测绝对化表述 ("必涨"/"稳赚" 等)，统计条件化表述 |
| 数据真实性 | 规则 | 报告中的价格/百分比能否在工具返回中溯源 (反幻觉) |
| 端到端延迟 / 工具调用效率 | 埋点 | 耗时、HTTP 请求数、缓存命中率、各工具调用次数 |
| 语义质量 (矛盾信号处理等) | LLM Judge | 评审 Agent 按 5 维量表打分 |

In [ ]:
# 1) 规则化考核: 对上面生成的综合报告做结构/条件化检查
from src.evaluation import evaluate_report, format_evaluation

evaluation = evaluate_report(comprehensive_report)
print(format_evaluation(evaluation))

In [ ]:
# 2) 带埋点的完整考核: 延迟 + 工具调用统计 + 数字溯源
#    (会重新跑一轮完整分析，消耗 LLM Token)
from src.evaluation import ToolCallCounter, timed_run, format_metrics
from src.tools.market_data import clear_cache

counter = ToolCallCounter()
coordinator_eval = create_coordinator(llm=llm, tool_counter=counter)
clear_cache()  # 清空缓存，统计真实请求数

report, metrics = timed_run(
    coordinator_eval,
    "请对 BTC 进行全面的综合分析，给出交叉验证后的综合判断和条件化建议。"
)

print(format_metrics(metrics, counter))
print()
# 数字溯源: 报告中的数字必须来自工具返回 (counter.outputs 记录了所有工具原始输出)
print(format_evaluation(evaluate_report(report, tool_outputs=counter.outputs)))

In [ ]:
# 3) LLM-as-Judge: 语义层面的质量评审 (矛盾信号处理、可执行性等)
from src.evaluation.judge import create_judge_agent, run_judge

judge = create_judge_agent(llm)
verdict = run_judge(judge, comprehensive_report)

if verdict["parse_ok"]:
    for k, v in verdict["scores"].items():
        print(f"{k}: {v}")
else:
    print("评分解析失败，原始输出:")
    print(verdict["raw"])

---
## Part 4.6: 信号留痕与历史胜率 (Track Record)

商业化的基石之一: **可验证的效果记录**。每次综合分析自动归档
(币种、方向判断、信号时刻价格)，到期后用真实历史价格核算对错，
积累可对外展示的胜率统计。

- 信号存储在 `outputs/signals.jsonl`，追加写、可审计
- 24h / 7d 两个核算周期；中性信号以 ±3% 波动带为判对标准
- 价格来自 Binance 历史 K 线，核算结果不可篡改地基于真实行情

In [ ]:
# 归档本次分析信号 (方向判断自动从报告提取，价格实时记录)
from src.evaluation import record_signal, update_outcomes, summarize_signals, format_signal_summary

entry = record_signal("BTC", comprehensive_report)
print(f"✅ 信号已归档: {entry['symbol']} {entry['bias']} @ ${entry['price_at_signal']:,.2f}")
print(f"   id={entry['id']}")

In [ ]:
# 核算到期信号并查看历史胜率
# (信号需 24h 后才有第一次核算结果；建议每天运行一次)
n = update_outcomes()
print(f"本次核算 {n} 条信号结果\n")
print(format_signal_summary(summarize_signals()))

---
## Part 5: 更多使用场景

In [ ]:
# 场景 1: 针对特定维度的深入分析
print("=" * 60)
print("📊 场景 1: ETH 技术面深入分析")
print("=" * 60)

eth_tech = coordinator.run(
    "我想重点从技术面分析 ETH 当前是否适合入场做多，"
    "请技术分析师给出详细的技术面判断，"
    "同时让情绪分析师确认一下当前情绪是否支持做多。"
)
print(eth_tech)

In [ ]:
# 场景 2: 风险评估
print("=" * 60)
print("⚠️ 场景 2: SOL 风险评估")
print("=" * 60)

sol_risk = coordinator.run(
    "我持有 SOL 的多头仓位，请帮我评估当前的风险水平。"
    "重点关注: 是否有需要警惕的信号？是否应该减仓或设置止损？"
)
print(sol_risk)

---
## Part 6: 架构说明与设计思考

### 6.1 为什么选择多 Agent 架构？

本项目的设计灵感来源于 NOFXi 交易系统的实战经验。在 NOFXi 中，我们发现:

1. **单 Agent 的局限性**: 当工具过多、Prompt 过长时，模型的决策质量会下降
2. **专业分工的价值**: 每个分析维度有独立的数据源和分析逻辑，适合独立处理
3. **Skill-first 思想**: 高频分析任务应该有稳定的执行路径，而不是每次重新规划

### 6.2 Agent 范式选择

| Agent | 范式 | 原因 |
|-------|------|------|
| 技术分析师 | ReAct | 需要观察数据→思考含义→决定下一步 |
| 链上分析师 | ReAct | 同上，链上数据需要逐步探索 |
| 情绪分析师 | ReAct | 同上，多个情绪指标需要逐一获取 |
| 综合协调员 | Plan-and-Solve | 任务明确，先规划再执行 |

### 6.3 与 NOFXi 架构的关系

NOFXi 的核心设计原则:
- **80% Skill + 20% 动态规划**: 高频任务固化为 Skill，只有复杂问题才动态规划
- **条件化建议**: 不做绝对预测，给出条件化的可执行建议
- **风险优先**: 所有操作都考虑风险，高风险动作需要确认

本项目将这些原则应用到了分析场景:
- 每个 Agent 的分析流程是固定的 (Skill 化)
- 输出格式是结构化的 (可预期)
- 建议是条件化的 (不做绝对判断)
- 风险提示是必须的 (安全第一)

---
## Part 7: 总结与展望

### 已实现功能

- ✅ 技术分析 Agent: K线、指标、支撑阻力
- ✅ 链上分析 Agent: 资金流向、巨鲸、活跃度
- ✅ 情绪分析 Agent: 恐惧贪婪、费率、社交
- ✅ 综合协调 Agent: 多维度交叉验证
- ✅ 结构化报告输出
- ✅ 条件化建议生成

### 遇到的挑战

1. **链上数据获取**: 大部分链上数据 API 需要付费，本项目使用了估算 + 公开 API 的组合方案
2. **多 Agent 协调**: 子 Agent 的输出格式需要标准化，否则协调器难以汇总
3. **Token 消耗**: 多 Agent 调用会显著增加 Token 消耗，需要控制每个 Agent 的输出长度

### 未来改进方向

- [ ] 接入真实链上数据源 (Glassnode, Dune)
- [ ] 添加 Reflection 机制，让 Agent 自我评估分析质量
- [ ] 支持定时分析，生成每日报告
- [ ] 添加回测模块，验证分析信号的历史表现
- [ ] 集成 MCP 协议，支持与 NOFXi 等系统互操作
- [ ] 优化 Token 使用，通过缓存减少重复调用